In [1]:
!pip install -U -q transformers datasets bitsandbytes trl peft huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 99.0 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 646.8/646.8 kB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.7 MB/s eta 0:00:00:00:0100:01


In [3]:
from huggingface_hub import notebook_login
notebook_login()

DPO

In [4]:
# Prompt          [PAD, PAD, 1, 2]        (left-pad)
# chosen          [3, 4, 5, PAD]          (right-pad)
# rejected        [6, PAD, PAD, PAD]      (right-pad)

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"
config_8bit = BitsAndBytesConfig(load_in_8bit=True)
model_4bit = AutoModelForCausalLM.from_pretrained(
                                                    model_name,
                                                    quantization_config=config_8bit,
                                                    device_map="auto",
                                                    trust_remote_code=True
                                                 )

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)                                                

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [6]:
from datasets import load_dataset

dataset = load_dataset("HumanLLMs/Human-Like-DPO-Dataset", split="train[:300]")
dataset

README.md: 0.00B [00:00, ?B/s]

data.json:   0%|          | 0.00/28.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10884 [00:00<?, ? examples/s]

Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 300
})

In [7]:
dataset[0]

{'prompt': 'Oh, I just saw the best meme - have you seen it?',
 'chosen': "😂 Ah, no I haven't! I'm dying to know, what's the meme about? Is it a funny cat or a ridiculous situation? Spill the beans! 🤣",
 'rejected': "I'm an artificial intelligence language model, I don't have personal experiences or opinions. However, I can provide you with information on highly-rated and critically acclaimed films, as well as recommendations based on specific genres or themes. Would you like me to suggest some notable movies or discuss a particular genre of interest?"}

In [8]:
example_test = [{'role':'assistant', 'content':dataset[0]['chosen']}]
example_test

[{'role': 'assistant',
  'content': "😂 Ah, no I haven't! I'm dying to know, what's the meme about? Is it a funny cat or a ridiculous situation? Spill the beans! 🤣"}]

In [9]:
tokenizer.apply_chat_template(example_test, tokenize=False)

"<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>assistant\n😂 Ah, no I haven't! I'm dying to know, what's the meme about? Is it a funny cat or a ridiculous situation? Spill the beans! 🤣<|im_end|>\n"

In [10]:
def apply_chat_temp(example):

    example['prompt'] = [{'role':'user', 'content':example['prompt']}]
    return example

new_dataset = dataset.map(apply_chat_temp)
new_dataset[0]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

{'prompt': [{'role': 'user',
   'content': 'Oh, I just saw the best meme - have you seen it?'}],
 'chosen': "😂 Ah, no I haven't! I'm dying to know, what's the meme about? Is it a funny cat or a ridiculous situation? Spill the beans! 🤣",
 'rejected': "I'm an artificial intelligence language model, I don't have personal experiences or opinions. However, I can provide you with information on highly-rated and critically acclaimed films, as well as recommendations based on specific genres or themes. Would you like me to suggest some notable movies or discuss a particular genre of interest?"}

In [11]:
new_template = """<|im_start|>assistant\n{model_answer}<|im_end|>\n"""

def format_dataset(example):

    example['prompt'] = tokenizer.apply_chat_template(example['prompt'],tokenize=False)
    example['chosen'] = new_template.format(model_answer=example['chosen'])
    example['rejected'] = new_template.format(model_answer=example['rejected'])

    return example

formatted_dataset = new_dataset.map(format_dataset)
formatted_dataset[0]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

{'prompt': '<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nOh, I just saw the best meme - have you seen it?<|im_end|>\n',
 'chosen': "<|im_start|>assistant\n😂 Ah, no I haven't! I'm dying to know, what's the meme about? Is it a funny cat or a ridiculous situation? Spill the beans! 🤣<|im_end|>\n",
 'rejected': "<|im_start|>assistant\nI'm an artificial intelligence language model, I don't have personal experiences or opinions. However, I can provide you with information on highly-rated and critically acclaimed films, as well as recommendations based on specific genres or themes. Would you like me to suggest some notable movies or discuss a particular genre of interest?<|im_end|>\n"}

In [12]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r = 32, 
    lora_alpha = 32, #the higher value will be , lora matrix will get influence than matrix of base model
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "down_proj", "up_proj"],
    lora_dropout = 0.05,
    bias = "none",
    task_type = "CAUSAL_LM"
)

lora_model = get_peft_model(model_4bit, lora_config)

In [13]:
lora_model.print_trainable_parameters()

trainable params: 36,929,536 || all params: 1,580,643,840 || trainable%: 2.3364


In [14]:
lora_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear8bitLt(
                (base_layer): Linear8bitLt(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): 

In [15]:
!pip install shutil

ERROR: Could not find a version that satisfies the requirement shutil (from versions: none)
ERROR: No matching distribution found for shutil


In [16]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [17]:
from trl import DPOTrainer, DPOConfig 

dpo_config = DPOConfig(
    output_dir = "/content/drive/MyDrive/results1",  # full absolute path
    per_device_train_batch_size = 2,
    max_steps = 40,
    gradient_accumulation_steps = 4,
    learning_rate = 3e-5,
    lr_scheduler_type = "cosine",
    optim = 'adamw_torch',
    logging_steps = 10,
    beta = 0.1,
    report_to = "none"
)



In [18]:
# Step 2: Check if mount worked and see the structure
import os

print("Drive contents:")
print(os.listdir('/content/drive/MyDrive'))

Drive contents:
['Getting started.pdf', 'IMG-20190811-WA0003.jpg', "I am sharing 'UID27220900156_03_C.xls.xlsx.xlsx' with you from WPS Office", 'IMG_20201221_133012.jpg', 'WhatsApp Image 2022-06-07 at 12.54.01 AM.jpeg', 'adhar.jpeg', 'ph-removebg-preview (1).png', 'Agriculture_Bionomial Brain_Crop yielding automation System.pptx', 'Agriculture_Binomial Brain_Crop yielding automation System (1).pptx', 'Agriculture_Binomial Brain_Crop yielding automation System.pptx', 'Screenshot_20230327_201205_Gmail.jpg', 'Screenshot_20230328_203720_Gmail.jpg', 'Resume_Yahya (1).pdf', 'Resume_Yahya.pdf', 'Daily Routine and Goals.gsheet', 'photo.jpg', 'Untitled spreadsheet (1).gsheet', '9.m4a', 'Voice 012.m4a', 'Voice 013.m4a', 'IMG-20220422-WA0044.jpg', 'IMG-20220422-WA0071.jpg', 'daily timeline & to-do list.gsheet', 'Colab Notebooks', 'photo (1).jpg', 'Questions.gsheet', 'Yahya Res (1).pdf', 'Yahya Res.pdf', 'ACPCE_YAHYA_CSEIOT.pdf', 'GSOC.gsheet', 'Untitled document.gdoc', 'Yahya_Resume .pdf', 'Repor

In [19]:
# Step 3: Manually create the folder first, then point DPO to it
save_path = "/content/drive/MyDrive/results1"
os.makedirs(save_path, exist_ok=True)
print(f"Created: {save_path}")
print(f"Exists: {os.path.exists(save_path)}")

Created: /content/drive/MyDrive/results1
Exists: True


In [21]:
trainer = DPOTrainer(
    model = lora_model,
    train_dataset = formatted_dataset,
    processing_class = tokenizer,
    args = dpo_config
)


Adding EOS to train dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/300 [00:00<?, ? examples/s]